#Speech-to-Reasoning Pipeline (Whisper + Unsloth Dynamic 4-bit LLM)

**Pipeline:** Audio -> Whisper ASR -> transcript -> Qwen3-4B (Unsloth dynamic 4-bit, reasoning/thinking mode) -> reasoning trace + final answer


**Memory strategy:** Whisper runs first in fp16 with batched, chunked inference. It is then deleted and the CUDA cache cleared *before* the LLM is loaded, so both models never compete for VRAM. The LLM is a dynamic 4-bit quant, and generation is batched with left-padding.

## 1. Install dependencies

In [1]:
%%capture
!pip install unsloth
!pip install -U transformers accelerate gTTS librosa soundfile

## 2. Imports and GPU helpers

In [2]:
import unsloth
import torch, gc, time, os
import numpy as np, librosa

assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory/1024**3:.2f} GB")

def vram(tag=""):
    print(f"[VRAM] {tag:<32} allocated={torch.cuda.memory_allocated()/1024**3:5.2f} GB | "
          f"peak={torch.cuda.max_memory_allocated()/1024**3:5.2f} GB")

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

vram("baseline")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4 | VRAM: 14.56 GB
[VRAM] baseline                         allocated= 0.01 GB | peak= 0.01 GB


## 3. Prepare sample audio queries



In [3]:
from gtts import gTTS

os.makedirs("/content/audio", exist_ok=True)
spoken_queries = {
    "q1_math.mp3":  "A train travels one hundred and twenty kilometres in two hours, and then one hundred and eighty kilometres in three hours. What is its average speed for the whole journey?",
    "q2_logic.mp3": "All roses are flowers, and some flowers fade quickly. Can we conclude that some roses fade quickly? Explain your reasoning.",
    "q3_qa.mp3":    "Why does four bit quantization let a large language model fit on a small GPU?",
}
audio_paths = []
for fname, text in spoken_queries.items():
    p = f"/content/audio/{fname}"
    gTTS(text=text, lang="en").save(p)
    audio_paths.append(p)
print(audio_paths)

from IPython.display import Audio, display
for p in audio_paths:
    print(os.path.basename(p)); display(Audio(p))

['/content/audio/q1_math.mp3', '/content/audio/q2_logic.mp3', '/content/audio/q3_qa.mp3']
q1_math.mp3


q2_logic.mp3


q3_qa.mp3


## 4. Stage 1: Whisper speech recognition



In [4]:
from transformers import pipeline

WHISPER_ID = "openai/whisper-small"

asr = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_ID,
    torch_dtype=torch.float16,
    device="cuda:0",
)
vram("after loading Whisper")

def load_audio(path, sr=16000):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    return {"raw": wav.astype(np.float32), "sampling_rate": sr}

def transcribe(paths, batch_size=4):
    inputs = [load_audio(p) for p in paths]
    t0 = time.time()
    results = asr(
        inputs,
        batch_size=batch_size,
        chunk_length_s=30,
        generate_kwargs={"language": "english", "task": "transcribe"},
    )
    print(f"Transcribed {len(paths)} file(s) in {time.time()-t0:.1f}s")
    return [r["text"].strip() for r in results]

transcripts = transcribe(audio_paths)
for p, t in zip(audio_paths, transcripts):
    print(f"\n{os.path.basename(p)}\n  -> {t}")
vram("after transcription")

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[VRAM] after loading Whisper            allocated= 0.46 GB | peak= 0.46 GB


Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set cle

Transcribed 3 file(s) in 14.6s

q1_math.mp3
  -> A train travels 120 km in 2 hours and then 180 km in 3 hours. What is its average speed for the whole journey?

q2_logic.mp3
  -> All roses are flowers. And some flowers fade quickly. Can we conclude that some roses fade quickly? Explain your reasoning.

q3_qa.mp3
  -> Why does 4-bit quantization let a large language model fit on a small GPU?
[VRAM] after transcription              allocated= 0.46 GB | peak= 1.33 GB


### Free Whisper before loading the LLM (important for small GPUs)

In [5]:
del asr
free_gpu()
vram("after freeing Whisper")

[VRAM] after freeing Whisper            allocated= 0.01 GB | peak= 1.33 GB


## 5. Stage 2: Load the Unsloth dynamic 4-bit reasoning model

`unsloth/Qwen3-4B-unsloth-bnb-4bit` is a dynamic 4-bit quant of Qwen3-4B, which supports a *thinking* mode (step-by-step reasoning inside `<think>...</think>`).
Alternatives: `unsloth/Qwen3-8B-unsloth-bnb-4bit`, `unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit`.

In [6]:
from unsloth import FastLanguageModel

LLM_ID = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLM_ID,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
vram("after loading dynamic 4-bit LLM")

==((====))==  Unsloth 2026.10.3: Fast Qwen3 patching. Transformers: 5.19.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[VRAM] after loading dynamic 4-bit LLM  allocated= 3.35 GB | peak= 3.37 GB


## 6. Batched reasoning function

* Each transcript becomes a chat message and is formatted with the model's chat template (UTF-8 safe, handles special tokens).
* Prompts are **left-padded** and generated together in batches to use the GPU efficiently.
* Qwen3 sampling settings for thinking mode: `temperature=0.6, top_p=0.95, top_k=20`.
* The output is split into the reasoning trace (`<think>`) and the final answer.

In [7]:
SYSTEM = "You are a careful reasoning assistant. The user's question was transcribed from speech, so ignore minor transcription errors. Think step by step, then give a clear final answer."

def split_thinking(text):
    if "</think>" in text:
        thought, answer = text.split("</think>", 1)
        return thought.replace("<think>", "").strip(), answer.strip()
    return "", text.strip()

@torch.inference_mode()
def reason(transcripts, batch_size=3, max_new_tokens=1024, thinking=True):
    results = []
    for i in range(0, len(transcripts), batch_size):
        batch = transcripts[i:i+batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM}, {"role": "user", "content": t}],
                tokenize=False, add_generation_prompt=True, enable_thinking=thinking,
            ) for t in batch
        ]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to("cuda")
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.6, top_p=0.95, top_k=20,
            use_cache=True, pad_token_id=tokenizer.pad_token_id,
        )
        gen = out[:, enc["input_ids"].shape[1]:]
        for row in gen:
            text = tokenizer.decode(row, skip_special_tokens=True)
            results.append(split_thinking(text))
        del enc, out, gen
        free_gpu()
    return results

## 7. End-to-end demo: audio in, reasoning out

In [8]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
outputs = reason(transcripts, batch_size=3)
print(f"Reasoning finished in {time.time()-t0:.1f}s\n")

for path, tr, (thought, answer) in zip(audio_paths, transcripts, outputs):
    print("="*90)
    print(f"AUDIO      : {os.path.basename(path)}")
    print(f"TRANSCRIPT : {tr}")
    print(f"\nREASONING (first 600 chars):\n{thought[:600]}{'...' if len(thought)>600 else ''}")
    print(f"\nFINAL ANSWER:\n{answer}")
vram("after reasoning")

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reasoning finished in 109.2s

AUDIO      : q1_math.mp3
TRANSCRIPT : A train travels 120 km in 2 hours and then 180 km in 3 hours. What is its average speed for the whole journey?

REASONING (first 600 chars):
Okay, let's see. The problem is about finding the average speed of a train for the whole journey. The train travels two different segments: first 120 km in 2 hours, then 180 km in 3 hours. I need to find the average speed for the entire trip.

Hmm, average speed. I remember that average speed is total distance divided by total time. Right? So I need to calculate the total distance the train traveled and the total time it took, then divide those two numbers.

Let me break it down. First segment: 120 km in 2 hours. Second segment: 180 km in 3 hours. So total distance would be 120 + 180. Let me a...

FINAL ANSWER:
The average speed for the whole journey is calculated by dividing the total distance by the total time. 

**Total distance** = 120 km + 180 km = 300 km  
**Total time** = 2

### Wrap everything as one function (audio files -> answers)

In [9]:
def speech_to_reasoning(paths, batch_size=3):
    global asr
    asr = pipeline("automatic-speech-recognition", model=WHISPER_ID, torch_dtype=torch.float16, device="cuda:0")
    texts = transcribe(paths, batch_size=batch_size)
    del asr; free_gpu()
    answers = reason(texts, batch_size=batch_size)
    return [{"audio": p, "transcript": t, "reasoning": r, "answer": a} for p, t, (r, a) in zip(paths, texts, answers)]

demo = speech_to_reasoning([audio_paths[0]])
print(demo[0]["transcript"], "\n->", demo[0]["answer"])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Transcribed 1 file(s) in 0.8s


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A train travels 120 km in 2 hours and then 180 km in 3 hours. What is its average speed for the whole journey? 
-> The average speed for the whole journey is calculated by dividing the total distance by the total time. 

**Total distance** = 120 km + 180 km = 300 km  
**Total time** = 2 hours + 3 hours = 5 hours  

**Average speed** = Total distance / Total time = 300 km / 5 hours = **60 km/h**.

**Answer:** The average speed for the whole journey is **60 km/h**.


## 8. Try your own audio

Upload an mp3/wav/m4a file with a spoken question (record one on your phone).

In [11]:
import glob, subprocess, gc, torch


try:
    del asr
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()


src = glob.glob("/content/audio/WhatsApp*")[0]
dst = "/content/audio/my_voice.wav"
subprocess.run(["ffmpeg", "-y", "-i", src, "-vn", "-ar", "16000", "-ac", "1", dst,
                "-loglevel", "error"], check=True)

from IPython.display import Audio, display
display(Audio(dst))

res = speech_to_reasoning([dst])[0]
print("TRANSCRIPT:", res["transcript"])
print("\nREASONING (first 500 chars):\n", res["reasoning"][:500])
print("\nFINAL ANSWER:\n", res["answer"])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Transcribed 1 file(s) in 0.7s


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TRANSCRIPT: If I have 5 apples and give away 2, then buy 4 more. How many apples do I have?

REASONING (first 500 chars):
 Okay, let's see. The user starts with 5 apples. Then they give away 2. So, first step is subtracting 2 from 5. That would be 5 minus 2 equals 3. Then they buy 4 more apples. So adding 4 to the remaining 3. That would be 3 plus 4. Let me check that again. 3 plus 4 is 7. So the total number of apples should be 7. Wait, is there any chance I missed something? Let me go through it again. Starting with 5, giving away 2 leaves 3. Buying 4 more adds to that. Yes, 3 plus 4 is definitely 7. I don't think

FINAL ANSWER:
 You start with 5 apples. After giving away 2, you have $5 - 2 = 3$ apples. Then, buying 4 more adds $3 + 4 = 7$ apples. 

**Final Answer:** 7 apples.


## 9. Memory summary

In [12]:
free_gpu()
vram("final")
print(f"Peak = {torch.cuda.max_memory_allocated()/1024**3:.2f} GB / {props.total_memory/1024**3:.1f} GB total")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

[VRAM] final                            allocated= 3.47 GB | peak= 4.60 GB
Peak = 4.60 GB / 14.6 GB total
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 3743 MiB, 15360 MiB
